<a href="https://colab.research.google.com/github/Hang8adntf/Hang8adntf_Programming_BigDataAnalytics/blob/main/3_02_key_value_stores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=1vv_PsWBnUJwSCkwKDoJAC-vXjtaEA4Ts)

# 3.02 Key-value Stores with TinyDB
This tutorial gives a basic introduction to working with key-value (KV) stores (or document DBs). We will be working with [TinyDB](https://tinydb.readthedocs.io/en/latest/index.html), an in-memory Python database, which is particularly attractive here as it is, as the name suggests, pretty small and lightweight.

We will begin with the relevant installs:

In [ ]:
!pip install tinydb
!pip install faker
!pip install python-lorem

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 24.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for python-lorem: filename=python_lorem-1.3.0.post3-py3-none-any.whl size=9166 sha256=9032d20a7387421386efb62f2aba706fe725d36160a5b6635468bd31c332ec81
  Stored in directory: /root/.cache/pip/wheels/9d/3a/d2/ba74831189552ab46d4b3d52d3ef86e4f06feef77b0a35d0ea
Successfully built python-lorem


As you may infer from the pacakages installed, we will run something similar to one of our DuckDB examples - specifically building a database using fake data generated by Faker:

In [ ]:
import random
from faker import Faker
import pandas as pd
from lorem import paragraph
import itertools

fake = Faker()

def get_person():
  person = {}
  person['id'] = random.randrange(1000,9999999999999)
  person['first_name'] = fake.first_name()
  person['last_name'] = fake.last_name()
  person['email'] = fake.unique.ascii_email()
  person['company'] = fake.company()
  person['phone'] = fake.phone_number()
  person['review'] = list(itertools.islice(paragraph(count=1), 1))
  return person

personlist = []
for x in range(100):
  personlist.append(get_person())

df = pd.DataFrame.from_dict(personlist)
df.head()

,id,first_name,last_name,email,company,phone,review
0,950256516142,Jamie,Stewart,dmiles@martinez-santos.com,Boyle and Sons,485-491-6497x0270,[Proident culpa sed deserunt officia sint pari...
1,2471635006339,Linda,Mcclure,townsendcaitlyn@dominguez-stanley.info,Munoz-Monroe,(839)497-2559x2158,"[Aute cillum minim officia, id consectetur nos..."
2,8527300701848,Jennifer,Barron,henrydavis@gmail.com,Williams PLC,504.355.3385x886,"[Enim do quis eu enim, et deserunt enim sunt i..."
3,7195538647314,Cody,Zimmerman,patrick13@gmail.com,Walton Inc,469-712-5230x072,[Sint duis velit est eiusmod. Cillum pariatur ...
4,8515016272320,Jaime,Williams,adam18@lynch.com,Ramirez-Rodriguez,589-314-2772x97308,[Do anim sed aliquip ullamco do sed. Qui ullam...


Everything here is the same except we have also add a text column (using lorem ipsum). As before we have created this as a Pandas dataframe, but like most KV stores, TinyDB prefers data stored as a dictionary:

In [ ]:
fake_data = df.to_dict(orient='records')
fake_data

[{'id': 950256516142,
  'first_name': 'Jamie',
  'last_name': 'Stewart',
  'email': 'dmiles@martinez-santos.com',
  'company': 'Boyle and Sons',
  'phone': '485-491-6497x0270',
  'review': ['Proident culpa sed deserunt officia sint pariatur. Tempor sunt esse adipiscing elit sunt, irure aliqua lorem in excepteur irure, velit esse occaecat est nulla. Minim commodo ex esse excepteur laboris. Duis dolor tempor nostrud proident excepteur officia. Exercitation veniam magna aliquip commodo. Consequat tempor sint fugiat non consequat do consequat, esse nulla ipsum enim consequat exercitation, sit dolor esse deserunt magna commodo consequat nisi. Nulla et sit ut pariatur labore voluptate, cupidatat ullamco id officia in lorem.']},
 {'id': 2471635006339,
  'first_name': 'Linda',
  'last_name': 'Mcclure',
  'email': 'townsendcaitlyn@dominguez-stanley.info',
  'company': 'Munoz-Monroe',
  'phone': '(839)497-2559x2158',
  'review': ['Aute cillum minim officia, id consectetur nostrud cupidatat sunt 

With this transform in place we can load the data into our database. You may note the database itself is specified as JSON format:

In [ ]:
from tinydb import TinyDB, Query

db = TinyDB('db.json')

for record in fake_data:
  db.insert(record)

We can check this has worked with a simple Python loop:

In [ ]:
for item in db:
  print(item)

{'id': 950256516142, 'first_name': 'Jamie', 'last_name': 'Stewart', 'email': 'dmiles@martinez-santos.com', 'company': 'Boyle and Sons', 'phone': '485-491-6497x0270', 'review': ['Proident culpa sed deserunt officia sint pariatur. Tempor sunt esse adipiscing elit sunt, irure aliqua lorem in excepteur irure, velit esse occaecat est nulla. Minim commodo ex esse excepteur laboris. Duis dolor tempor nostrud proident excepteur officia. Exercitation veniam magna aliquip commodo. Consequat tempor sint fugiat non consequat do consequat, esse nulla ipsum enim consequat exercitation, sit dolor esse deserunt magna commodo consequat nisi. Nulla et sit ut pariatur labore voluptate, cupidatat ullamco id officia in lorem.']}
{'id': 2471635006339, 'first_name': 'Linda', 'last_name': 'Mcclure', 'email': 'townsendcaitlyn@dominguez-stanley.info', 'company': 'Munoz-Monroe', 'phone': '(839)497-2559x2158', 'review': ['Aute cillum minim officia, id consectetur nostrud cupidatat sunt enim nisi. Ut occaecat occa

With our database setup, we can start to query our records. In TinyDB we do this by creating a query object:

In [ ]:
User = Query() # query object

db.search(User.first_name == 'Jamie') # adapt based on your data

[{'id': 950256516142,
  'first_name': 'Jamie',
  'last_name': 'Stewart',
  'email': 'dmiles@martinez-santos.com',
  'company': 'Boyle and Sons',
  'phone': '485-491-6497x0270',
  'review': ['Proident culpa sed deserunt officia sint pariatur. Tempor sunt esse adipiscing elit sunt, irure aliqua lorem in excepteur irure, velit esse occaecat est nulla. Minim commodo ex esse excepteur laboris. Duis dolor tempor nostrud proident excepteur officia. Exercitation veniam magna aliquip commodo. Consequat tempor sint fugiat non consequat do consequat, esse nulla ipsum enim consequat exercitation, sit dolor esse deserunt magna commodo consequat nisi. Nulla et sit ut pariatur labore voluptate, cupidatat ullamco id officia in lorem.']}]

We can also add new data in dictionary/JSON-like format:

In [ ]:
db.insert({'id': 123, 'first_name': 'Amir', 'star_sign': 'Dog', 'review': 'I do not speak Latin.'})

101

And retrieve the data as before:

In [ ]:
db.search(User.id == 123)

[{'id': 123,
  'first_name': 'Amir',
  'star_sign': 'Dog',
  'review': 'I do not speak Latin.'}]

One thing to note here is that our new record does not follow the schema we may infer from the original dataset (i.e. the original data all used the same columns/fields). Here many of those fields are missing and we have the new field 'star_sign'.

This demonstrates the extra flexibility we get with a KV store over a relational model. We can also query our database to get all records that have a specific field:

In [ ]:
db.search(User.star_sign.exists())

[{'id': 123,
  'first_name': 'Amir',
  'star_sign': 'Dog',
  'review': 'I do not speak Latin.'}]

This gives a basic intro into KV (and document) stores. While there are many competing brands/solutions, the common themes are the dictionary-like structure (key-value pairs) and flexibility to accept any fields (keys).